In [1]:
import os
import mne
import tqdm
import numpy
import shutil
import joblib

ROOT = "E:/CourseProject/sleep-edf"
exp = "exp_sleep_30_sec"
event_id = {
    "Sleep stage W": 1,
    "Sleep stage 1": 2,
    "Sleep stage 2": 3,
    "Sleep stage 3": 4,
    "Sleep stage 4": 4,
    "Sleep stage R": 5
}
epoch_sec = 30.0

In [2]:
def process(subj):
    dir = f"{ROOT}/{subj}"
    raw_train = mne.io.read_raw_fif(f'{dir}\data.fif')
    raw_train.resample(25)
    events_train, _ = mne.events_from_annotations(raw_train, event_id=event_id, chunk_duration=epoch_sec, verbose='error')

    tmax = epoch_sec - 1.0 / raw_train.info["sfreq"]
    epochs_train = mne.Epochs(raw=raw_train, events=events_train, tmin=0.0, tmax=tmax, baseline=None, verbose='error')
    print(epochs_train.get_data().shape)

    event_types = epochs_train.events[:, 2]
    stage_edges = [0]
    for i in range(1, len(epochs_train)):
        if event_types[i - 1] != event_types[i]:
            stage_edges.append(i)
    stage_edges.append(len(epochs_train))
    stage_edges = numpy.array(stage_edges)

    dir = f"{dir}/{exp}"
    shutil.rmtree(dir, ignore_errors=True)
    os.makedirs(dir)
    epochs_train.save(f"{dir}/epochs.fif")
    numpy.savetxt(f"{dir}/edges.txt", stage_edges.astype(int))

In [3]:
tasks = joblib.Parallel(n_jobs = -1)(
    joblib.delayed(process)(f"Subj{i}") for i in tqdm.trange(0, 153)
)

100%|██████████| 153/153 [01:45<00:00,  1.46it/s]
